# Statistical Tests for Missing Data Mechanisms

This notebook implements the tests used to identify the missing data mechanism
(MCAR, MAR, MNAR) and runs each of them on three generated datasets whose
mechanism is known by construction.

## MCAR tests
1. Little's MCAR test
2. Permutation test on the missingness indicator

## MAR
1. Logistic regression on the missingness indicator

## MNAR
1. Pattern mixture sensitivity analysis over an assumed shift

**On the data.** The three datasets below are generated in this notebook and are
not `survey.csv` from the lecture slides. The numbers here will not match the
numbers on the slides. What carries over is the method: the same four tests,
written out so you can read what each one computes.


In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# For missing data visualization
import missingno as msno

# For statistical tests
from scipy import stats
from scipy.stats import chi2_contingency
#from statsmodels.stats.multivariate import multi_normal_loglike
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from statsmodels.discrete.discrete_model import Logit

# For imputation methods
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression

# For evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Create Sample Dataset

Let's recreate the same datasets used in the class demonstration.

In [2]:
# Create a sample dataset
n_samples = 1000

# Generate correlated variables
np.random.seed(42)
age = np.random.randint(12, 80, n_samples)
income = 50000 +  np.random.normal(0, 30000, n_samples)
education_years = 12 +  np.random.randint(0, 10, n_samples)
health_score =  np.random.randint(20, 100, n_samples)

# Create DataFrame
df_complete = pd.DataFrame({
    'age': age,
    'income': income,
    'education_years': education_years,
    'health_score': health_score,
    'gender': np.random.choice(['Male', 'Female'], n_samples),
    'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston'], n_samples)
})

# Ensure positive values where appropriate
df_complete['age'] = np.clip(df_complete['age'], 18, 80)
df_complete['income'] = np.clip(df_complete['income'], 20000, 200000)
df_complete['education_years'] = np.clip(df_complete['education_years'], 8, 20)
df_complete['health_score'] = np.clip(df_complete['health_score'], 20, 100)

print("Complete dataset created:")
print(df_complete.head())
print(f"\nDataset shape: {df_complete.shape}")
print(f"Missing values: {df_complete.isnull().sum().sum()}")

Complete dataset created:
   age        income  education_years  health_score  gender         city
0   63  76092.213207               18            35  Female      Houston
1   26  81713.748001               20            70  Female      Chicago
2   72  20000.000000               18            99  Female     New York
3   32  35916.832598               20            91    Male      Houston
4   35  20000.000000               14            31    Male  Los Angeles

Dataset shape: (1000, 6)
Missing values: 0


In [3]:
print(df_complete.describe())

               age         income  education_years  health_score
count  1000.000000    1000.000000      1000.000000    1000.00000
mean     45.434000   53955.363516        16.323000      60.60100
std      19.394891   26732.774896         2.731324      22.82496
min      18.000000   20000.000000        12.000000      20.00000
25%      28.000000   31444.682737        14.000000      41.00000
50%      45.000000   51240.137404        16.000000      61.00000
75%      63.000000   71979.994851        19.000000      80.00000
max      79.000000  145021.150162        20.000000      99.00000


### 1.1 Create MCAR Dataset

In [4]:
# Create MCAR missing data
df_mcar = df_complete.copy()

# Randomly introduce missing values (10% missing rate)
missing_rate = 0.1
for col in ['income', 'health_score']:
    missing_indices = np.random.choice(df_mcar.index,
                                     size=int(len(df_mcar) * missing_rate),
                                     replace=False)
    df_mcar.loc[missing_indices, col] = np.nan

print("MCAR Dataset - Missing values summary:")
print(df_mcar.isnull().sum())
print(f"\nTotal missing values: {df_mcar.isnull().sum().sum()}")
print(f"Missing percentage: {(df_mcar.isnull().sum().sum() / (len(df_mcar) * len(df_mcar.columns))) * 100:.2f}%")

print(f"Average income of people with complete income data: ${df_mcar[df_mcar['income'].notnull()]['income'].mean():.2f}")
print(f"Average health score of people with complete health data: {df_mcar[df_mcar['health_score'].notnull()]['health_score'].mean():.2f}")

MCAR Dataset - Missing values summary:
age                  0
income             100
education_years      0
health_score       100
gender               0
city                 0
dtype: int64

Total missing values: 200
Missing percentage: 3.33%
Average income of people with complete income data: $53762.87
Average health score of people with complete health data: 60.83


### 1.2 Create MAR Dataset

In [5]:
# Create MAR missing data
df_mar = df_complete.copy()

# Income is more likely to be missing for younger people
young_threshold = df_mar['age'].quantile(0.3)
young_indices = df_mar[df_mar['age'] < young_threshold].index
missing_young = np.random.choice(young_indices,
                               size=int(len(young_indices) * 0.33),
                               replace=False)
df_mar.loc[missing_young, 'income'] = np.nan

# Health score is more likely to be missing for males
male_indices = df_mar[df_mar['gender'] == 'Male'].index
missing_male = np.random.choice(male_indices,
                              size=int(len(male_indices) * 0.2),
                              replace=False)
df_mar.loc[missing_male, 'health_score'] = np.nan

print("MAR Dataset - Missing values summary:")
print(df_mar.isnull().sum())

# Analyze the relationship between missingness and observed variables
print("\nMissingness analysis:")
print(f"Average age of people with missing income: {df_mar[df_mar['income'].isnull()]['age'].mean():.2f}")
print(f"Average age of people with complete income: {df_mar[df_mar['income'].notnull()]['age'].mean():.2f}")

print(f"Average income of people with complete income data: ${df_mar[df_mar['income'].notnull()]['income'].mean():.2f}")
print(f"Average health score of people with complete health data: {df_mar[df_mar['health_score'].notnull()]['health_score'].mean():.2f}")

print(f"\nPercentage of males with missing health score: {(df_mar[(df_mar['gender'] == 'Male') & (df_mar['health_score'].isnull())].shape[0] / df_mar[df_mar['gender'] == 'Male'].shape[0]) * 100:.2f}%")
print(f"Percentage of females with missing health score: {(df_mar[(df_mar['gender'] == 'Female') & (df_mar['health_score'].isnull())].shape[0] / df_mar[df_mar['gender'] == 'Female'].shape[0]) * 100:.2f}%")

MAR Dataset - Missing values summary:
age                 0
income             97
education_years     0
health_score       98
gender              0
city                0
dtype: int64

Missingness analysis:
Average age of people with missing income: 22.14
Average age of people with complete income: 47.94
Average income of people with complete income data: $53608.87
Average health score of people with complete health data: 60.90

Percentage of males with missing health score: 20.00%
Percentage of females with missing health score: 0.00%


### 1.3 Create MNAR Dataset

In [6]:
# Create MNAR missing data
df_mnar = df_complete.copy()

# High earners are more likely to not report their income
high_income_threshold = df_mnar['income'].quantile(0.8)
high_income_indices = df_mnar[df_mnar['income'] > high_income_threshold].index
missing_high_income = np.random.choice(high_income_indices,
                                     size=int(len(high_income_indices) * 0.5),
                                     replace=False)
df_mnar.loc[missing_high_income, 'income'] = np.nan

# People with low health scores are more likely to not report them
low_health_threshold = df_mnar['health_score'].quantile(0.2)
low_health_indices = df_mnar[df_mnar['health_score'] < low_health_threshold].index
missing_low_health = np.random.choice(low_health_indices,
                                    size=int(len(low_health_indices) * 0.5),
                                    replace=False)
df_mnar.loc[missing_low_health, 'health_score'] = np.nan

print("MNAR Dataset - Missing values summary:")
print(df_mnar.isnull().sum())

# Analyze the relationship
print("\nMissingness analysis:")
print(f"Average income of people with complete income data: ${df_mnar[df_mnar['income'].notnull()]['income'].mean():.2f}")
print(f"Average health score of people with complete health data: {df_mnar[df_mnar['health_score'].notnull()]['health_score'].mean():.2f}")
print("\nNote: In MNAR, we can't directly observe the relationship since the missing values depend on the unobserved values themselves.")

MNAR Dataset - Missing values summary:
age                  0
income             100
education_years      0
health_score        98
gender               0
city                 0
dtype: int64

Missingness analysis:
Average income of people with complete income data: $49278.12
Average health score of people with complete health data: 64.20

Note: In MNAR, we can't directly observe the relationship since the missing values depend on the unobserved values themselves.


## 2. MCAR Tests

### 2.1 Little's MCAR Test

Little's MCAR test is a statistical test that examines whether data are missing completely at random. The null hypothesis is that the data are MCAR. If the p-value is less than the significance level (e.g., 0.05), we reject the null hypothesis and conclude that the data are not MCAR.

In [7]:
def littles_mcar_test(df, numeric_cols=None):
    """
    Little's MCAR test.

    Groups the rows by their missingness pattern, measures how far each
    group's mean sits from the overall mean in Mahalanobis distance, and
    adds those distances up.

    Degrees of freedom are sum(observed variables in each pattern) minus the
    number of variables, which is the count of free mean parameters the
    alternative adds over the MCAR null.

    Parameters:
    df (pandas.DataFrame): DataFrame with missing values
    numeric_cols (list): Numeric columns to include in the test

    Returns:
    tuple: (test statistic, p-value, degrees of freedom)
    """
    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    df_numeric = df[numeric_cols]

    # Mean and covariance estimated on all available data
    means = df_numeric.mean()
    cov_matrix = df_numeric.cov()

    # One group per distinct missingness pattern
    missing_patterns = df_numeric.isnull().astype(int)
    pattern_groups = missing_patterns.groupby(numeric_cols).groups

    d2 = 0.0
    observed_total = 0

    for pattern, indices in pattern_groups.items():
        observed_cols = [col for col, missing in zip(numeric_cols, pattern) if missing == 0]
        if not observed_cols:          # every value missing: nothing to compare
            continue

        pattern_data = df_numeric.loc[indices, observed_cols]
        n_pattern = len(pattern_data)
        pattern_means = pattern_data.mean()

        means_subset = means[observed_cols]
        cov_subset = cov_matrix.loc[observed_cols, observed_cols]

        mean_diff = pattern_means - means_subset
        try:
            cov_inv = np.linalg.inv(cov_subset)
        except np.linalg.LinAlgError:  # singular submatrix: pattern contributes nothing
            continue

        d2 += n_pattern * mean_diff.dot(cov_inv).dot(mean_diff)
        observed_total += len(observed_cols)

    df_test = observed_total - len(numeric_cols)
    if df_test <= 0:
        return d2, np.nan, df_test

    p_value = stats.chi2.sf(d2, df_test)   # sf, not 1 - cdf, which underflows to 0
    return d2, p_value, df_test


In [8]:
# Apply Little's MCAR test to our datasets
numeric_cols = ['age', 'income', 'education_years', 'health_score']

def report_little(name, df):
    d2, p, dfree = littles_mcar_test(df, numeric_cols)
    print(f"Little's MCAR test on the {name} dataset:")
    print(f"  chi2 = {d2:.4f}   df = {dfree}   p = {p:.4g}")
    print(f"  {'MCAR not rejected' if p > 0.05 else 'MCAR rejected'}\n")
    return p

little_p_mcar = report_little('MCAR', df_mcar)
little_p_mar  = report_little('MAR',  df_mar)
little_p_mnar = report_little('MNAR', df_mnar)


Little's MCAR test on the MCAR dataset:
  chi2 = 4.3700   df = 8   p = 0.8223
  MCAR not rejected

Little's MCAR test on the MAR dataset:
  chi2 = 162.3901   df = 8   p = 5.058e-31
  MCAR rejected

Little's MCAR test on the MNAR dataset:
  chi2 = 14.5933   df = 8   p = 0.06755
  MCAR not rejected



### 2.2 Permutation/Randomization Test for MCAR

This test compares the distribution of observed values between groups with and without missing values. If the data are MCAR, there should be no significant difference between these distributions.

In [9]:
def permutation_test_mcar(df, var_with_missing, var_to_compare, n_permutations=1000):
    """
    Permutation test to check if data are MCAR by comparing distributions

    Parameters:
    df (pandas.DataFrame): DataFrame with missing values
    var_with_missing (str): Column name with missing values
    var_to_compare (str): Column name to compare distributions
    n_permutations (int): Number of permutations for the test

    Returns:
    tuple: (observed difference, p-value)
    """
    # Create missingness indicator
    missing_indicator = df[var_with_missing].isnull().astype(int)

    # Get values to compare
    values_to_compare = df[var_to_compare].values

    # Calculate observed difference in means
    mean_missing = df.loc[missing_indicator == 1, var_to_compare].mean()
    mean_observed = df.loc[missing_indicator == 0, var_to_compare].mean()
    observed_diff = abs(mean_missing - mean_observed)

    # Permutation test
    permutation_diffs = []
    for _ in range(n_permutations):
        # Shuffle the missingness indicator
        shuffled_indicator = np.random.permutation(missing_indicator)

        # Calculate difference in means for shuffled data
        mean_missing_perm = values_to_compare[shuffled_indicator == 1].mean()
        mean_observed_perm = values_to_compare[shuffled_indicator == 0].mean()
        perm_diff = abs(mean_missing_perm - mean_observed_perm)

        permutation_diffs.append(perm_diff)

    # Calculate p-value. The +1 on both counts keeps it away from an exact
    # zero, which no permutation test can support: with B shuffles the
    # smallest reportable value is 1/(B+1).
    n_at_least = sum(diff >= observed_diff for diff in permutation_diffs)
    p_value = (1 + n_at_least) / (1 + n_permutations)

    return observed_diff, p_value


In [10]:
# A permutation test that uses both observed columns at once.
# The statistic sums the squared standardised mean gaps across the columns
# in Z, so a small shift in each column still adds up to a large total.

Z = df_mcar[['age', 'education_years']]      # fully observed numeric columns
M = df_mcar['income'].isna().astype(int)     # the missingness indicator

def test_statistic(M, Z):
    g1 = Z[M == 1].mean()
    g0 = Z[M == 0].mean()
    s2 = Z.var()
    return (((g1 - g0) ** 2) / s2).sum()

T_obs = test_statistic(M, Z)

# Permutation null: reshuffle who is missing, keep the columns fixed
B = 5000
T_perm = []
for b in range(B):
    M_perm = np.random.permutation(M)
    T_perm.append(test_statistic(M_perm, Z))

perm_p_multivar = (1 + np.sum(np.array(T_perm) >= T_obs)) / (1 + B)
print(f"Observed T: {T_obs:.4f}")
print(f"Permutation p: {perm_p_multivar:.4f}")


Observed T: 0.0249
Permutation p: 0.3245


In [11]:
# Apply the single-column permutation test to our datasets

def report_perm(name, df, var_with_missing, var_to_compare):
    diff, p = permutation_test_mcar(df, var_with_missing, var_to_compare)
    print(f"{name} dataset: is '{var_with_missing}' missingness related to '{var_to_compare}'?")
    print(f"  observed difference in means = {diff:.4f}   p = {p:.4f}")
    print(f"  {'MCAR not rejected' if p > 0.05 else 'MCAR rejected'}\n")
    return p

perm_p_mcar  = report_perm('MCAR', df_mcar, 'income', 'age')
perm_p_mcar2 = report_perm('MCAR', df_mcar, 'health_score', 'age')
perm_p_mar   = report_perm('MAR',  df_mar,  'income', 'age')
perm_p_mnar  = report_perm('MNAR', df_mnar, 'income', 'education_years')


MCAR dataset: is 'income' missingness related to 'age'?
  observed difference in means = 0.9844   p = 0.6424
  MCAR not rejected

MCAR dataset: is 'health_score' missingness related to 'age'?
  observed difference in means = 0.6156   p = 0.7532
  MCAR not rejected



MAR dataset: is 'income' missingness related to 'age'?
  observed difference in means = 25.7914   p = 0.0010
  MCAR rejected



MNAR dataset: is 'income' missingness related to 'education_years'?
  observed difference in means = 0.4078   p = 0.1558
  MCAR not rejected



## 3. MAR Tests



### 3.1 Logistic Regression Test for MAR

This test uses logistic regression to predict missingness based on observed variables. If any observed variables significantly predict missingness, it suggests that the data are MAR.

In [12]:
def logistic_regression_test(df, var_with_missing, predictors):
    """
    Fit a logistic regression of the missingness indicator on the observed
    columns. The likelihood-ratio p-value tests the whole model, which is the
    MCAR test. The Wald p beside each coefficient attributes the result to one
    predictor, given the others.

    Parameters:
    df (pandas.DataFrame): DataFrame with missing values
    var_with_missing (str): Column whose missingness is modelled
    predictors (list): Observed columns to use as predictors

    Returns:
    tuple: (fitted model, significant predictors by Wald p < 0.05)
    """
    y = df[var_with_missing].isnull().astype(int)

    X = df[predictors].copy()
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    if categorical_cols:
        X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    X = X.astype(float)
    X = add_constant(X)

    model = Logit(y, X).fit(disp=0)

    significant_predictors = model.pvalues[model.pvalues < 0.05].index.tolist()
    if 'const' in significant_predictors:
        significant_predictors.remove('const')

    print(model.summary())
    print(f"\nLikelihood-ratio chi2 = {model.llr:.2f} on {int(model.df_model)} df, p = {model.llr_pvalue:.4g}")
    return model, significant_predictors


In [13]:
# Apply the logistic regression test to our datasets
predictors = ['age', 'education_years', 'gender']

def report_logit(name, df, var_with_missing):
    print(f"===== {name} dataset: {var_with_missing} =====")
    model, sig = logistic_regression_test(df, var_with_missing, predictors)
    print(f"Significant predictors (Wald p < 0.05): {sig}")
    print(f"{'MCAR not rejected' if model.llr_pvalue > 0.05 else 'MCAR rejected'}\n")
    return model.llr_pvalue, sig

logit_p_mcar,  sig_mcar  = report_logit('MCAR', df_mcar, 'income')
logit_p_mar,   sig_mar   = report_logit('MAR',  df_mar,  'income')
logit_p_mar2,  sig_mar2  = report_logit('MAR',  df_mar,  'health_score')
logit_p_mnar,  sig_mnar  = report_logit('MNAR', df_mnar, 'income')


===== MCAR dataset: income =====
                           Logit Regression Results                           
Dep. Variable:                 income   No. Observations:                 1000
Model:                          Logit   Df Residuals:                      996
Method:                           MLE   Df Model:                            3
Date:                Thu, 17 Sep 2026   Pseudo R-squ.:                0.006167
Time:                        13:44:27   Log-Likelihood:                -323.08
converged:                       True   LL-Null:                       -325.08
Covariance Type:            nonrobust   LLR p-value:                    0.2604
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -3.4099      0.722     -4.720      0.000      -4.826      -1.994
age                 0.0024      0.005      0.446      0.656      -0.008       0.013

/home/arun/venv_langgraph/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


### 3.2 Read the two p-values separately, and watch for separation

Two things in the output above are worth stopping on.

**The MAR health_score model did not converge.** `gender_Male` comes back at
14.02 with a standard error of 96.31, and its Wald p is 0.884. That is
separation: health_score was made missing only for males, so every female row
has y = 0 and the coefficient on gender is trying to run to infinity. A
standard error in the tens or hundreds is the symptom to watch for.

**The likelihood-ratio p on that same model is 4.5e-34.** The column is
obviously not MCAR. The Wald p said nothing useful because the one predictor
carrying the signal could not be estimated.

So: **quote the likelihood-ratio p as the verdict on the column, and a Wald p
only for a predictor you are naming.** They answer different questions, and a
large Wald p next to a small LR p usually means either separation, as here, or
two predictors carrying the same signal.


## 4. MNAR : Pattern Mixture Model

Pattern mixture models stratify data based on missing data patterns and fit separate models for each pattern.

In [14]:
def pmm_mean_delta(df, target_var, deltas):
    """
    Simple pattern-mixture sensitivity analysis:
    Impute missing values as (observed mean + delta)
    and recompute overall mean for each delta.
    """
    observed = df[target_var].dropna()
    n = len(df)
    n_mis = df[target_var].isna().sum()
    n_obs = len(observed)
    mean_obs = observed.mean()

    results = []
    for d in deltas:
        # assign all missing as mean_obs + d
        total = n_obs * mean_obs + n_mis * (mean_obs + d)
        overall_mean = total / n
        results.append({
            "delta": d,
            "n_missing": n_mis,
            "mean_missing_assumed": mean_obs + d,
            "overall_mean": overall_mean
        })

    return pd.DataFrame(results)

In [15]:
deltas = [0, 20000, 40000]
sens = pmm_mean_delta(df_mnar, target_var='income', deltas=deltas)
print(sens)

   delta  n_missing  mean_missing_assumed  overall_mean
0      0        100          49278.120702  49278.120702
1  20000        100          69278.120702  51278.120702
2  40000        100          89278.120702  53278.120702


## 5. Summary and Conclusions

Let's summarize the results of our tests for each dataset.

In [16]:
# Summary: what each test reported on each dataset
def verdict(p):
    return f"p = {p:.4g} ({'MCAR not rejected' if p > 0.05 else 'MCAR rejected'})"

summary = pd.DataFrame(index=['MCAR dataset', 'MAR dataset', 'MNAR dataset'])

summary["Little's MCAR test"] = [
    verdict(little_p_mcar), verdict(little_p_mar), verdict(little_p_mnar)
]

summary['Permutation test'] = [
    verdict(perm_p_mcar), verdict(perm_p_mar), verdict(perm_p_mnar)
]

summary['Logistic regression, LR p'] = [
    verdict(logit_p_mcar), verdict(logit_p_mar), verdict(logit_p_mnar)
]

summary['Predictors with Wald p < 0.05'] = [
    ', '.join(sig_mcar) or 'none',
    ', '.join(sig_mar) or 'none',
    ', '.join(sig_mnar) or 'none'
]

summary['True mechanism (by construction)'] = ['MCAR', 'MAR', 'MNAR']

summary


,Little's MCAR test,Permutation test,"Logistic regression, LR p",Predictors with Wald p < 0.05,True mechanism (by construction)
MCAR dataset,p = 0.8223 (MCAR not rejected),p = 0.6424 (MCAR not rejected),p = 0.2604 (MCAR not rejected),none,MCAR
MAR dataset,p = 5.058e-31 (MCAR rejected),p = 0.000999 (MCAR rejected),p = 8.033e-46 (MCAR rejected),age,MAR
MNAR dataset,p = 0.06755 (MCAR not rejected),p = 0.1558 (MCAR not rejected),p = 0.2061 (MCAR not rejected),none,MNAR


## 6. Conclusion

All three tests share one null: **missingness does not depend on anything
observed**, which is MCAR. Read the table above with that in mind.

- **MCAR dataset.** No test rejects. Nothing observed predicts which values went
  missing, which is what the generating code did.
- **MAR dataset.** All three reject. Missingness was built to depend on `age`
  and `gender`, both of which sit in the file, so the tests can see it.
- **MNAR dataset.** **No test rejects.** Missingness was built to depend on the
  income and health values themselves, those values are absent from exactly the
  rows where they would matter, and nothing else in the file stands in for them.
  Every test reports a clean column. The bias is entirely intact.

That last row is the point of the notebook. The MNAR column is as biased as it
ever was, and running more tests on it produces more reassurance.

**No test here identifies MNAR, and none can.** MAR and MNAR differ only in a
column that is missing from exactly the rows where it matters. A rejection tells
you the column is not MCAR. It does not tell you whether the dependence stops at
the observed columns.

MNAR gets past the tests in two different ways, and it is worth seeing both:

1. **No observed proxy**, as here. The tests fail to reject, you treat the column
   as clean, and the bias stays where it is.
2. **An observed proxy**, as with `health_score` in the lecture, where age and
   health correlate at -0.54. The tests reject and point at age, you impute on
   age, you recover about a fifth of the bias, and you report a number that
   looks precise.

Neither case is detectable from the file. Which one you are in depends on a
correlation that has nothing to do with why the data went missing.

This is why section 4 does something different. A pattern mixture analysis does
not test anything. It puts the part you cannot identify into one named number,
the shift `delta` between the missing group and the observed group, and reports
the estimate as a function of it. `delta = 0` is the complete-case answer. The
range of `delta` you are willing to defend, argued from how the question was
asked and who declined it, is the honest output.

What the verdicts permit for the next step:

- **MCAR** permits complete-case analysis.
- **MAR** permits imputation or weighting on the observed columns that carry the
  dependence.
- **MNAR** permits a reported range, and nothing narrower without new data, such
  as a follow-up sample of the people who did not answer.
